# primitive-lean × Kaggle

GitHubの固定コミットと、Kaggle側の原本をハッシュで対応させます。まず設定を変えずに **Run All** でファイルの対応確認だけを実行してください。

1. Kaggleに既存の原本フォルダ、または同じ **archive.zip** をInputとして用意します。自動検出できなければ下の `SOURCE_PATH` を指定します。
2. 環境準備に進むときはNotebookの **InternetをOn**、`RUN_SETUP=True`。
3. 実際の検査に進むときは `RUN_SETUP=True` と `RUN_PROOFS=True`。CPUで実行します。

ローカル原本の利用にはGitHubトークン不要です。GitHubから直接取得する場合だけ `SOURCE_MODE='github'` とし、Kaggle Secretsに `GITHUB_TOKEN`（対象リポジトリのContents読み取り）を登録します。トークンをコードへ貼り付けません。

検査結果の `passed` は今回のLean証明・公理検査の範囲だけを表します。過去ZIPの保存監査・本文描画・配送監査は別です。NotebookはKaggleコンペへの提出やGitHubへの書き戻しを行いません。


In [ ]:
# ここだけ必要に応じて変更します
SOURCE_MODE = "auto"       # auto / path / github
SOURCE_PATH = ""           # 例: /kaggle/input/your-data/archive.zip または原本フォルダ
RUN_SETUP = False           # 初回はFalseのままファイル対応だけ確認
RUN_PROOFS = False          # 実ビルドする段階でTrue（RUN_SETUPもTrue）
REBUILD = False             # Trueでこのプロジェクトのビルド生成物だけ再作成

WORKSPACE = "/kaggle/working/primitive-lean"
SOURCE_SEARCH_ROOTS = ["/kaggle/working", "/kaggle/input"]
GITHUB_SECRET_LABEL = "GITHUB_TOKEN"
REUSE_EXISTING_CACHE = True


In [ ]:
"""Kaggle notebook support: select exact sources, prepare tools, record real runs.

The notebook embeds this module so an existing Kaggle archive can bootstrap
without GitHub credentials. It never uploads or pushes to GitHub.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile


def sha_bytes(value):
    return hashlib.sha256(value).hexdigest()


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", "utf-8")


def read_candidate(path, expected):
    """Read only known source paths; no general ZIP extraction is performed."""
    path = Path(path)
    if path.is_dir():
        return {p: (path / p).read_bytes() for p in expected if (path / p).is_file()}
    if not path.is_file() or not zipfile.is_zipfile(path):
        raise ValueError("Not a source directory or ZIP")
    if path.stat().st_size > 64 * 1024 * 1024:
        raise ValueError("Use a source-only ZIP smaller than 64 MiB")
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("Duplicate ZIP entries")
        suffix = "lean/lean-toolchain"
        prefixes = [n[:-len(suffix)] for n in names if n.endswith(suffix)]
        if len(prefixes) != 1:
            raise ValueError("ZIP must contain exactly one Lean project root")
        prefix = prefixes[0]
        result = {}
        for rel in expected:
            name = prefix + rel
            if name in names:
                info = archive.getinfo(name)
                if info.file_size > 4 * 1024 * 1024:
                    raise ValueError("Unexpected oversized source file")
                result[rel] = archive.read(info)
        return result


def source_matches(files, expected, overlays):
    return [p for p, digest in expected.items()
            if p not in files or sha_bytes(files[p]) not in
            {digest, sha_bytes(overlays[p].encode("utf-8")) if p in overlays else digest}]


def find_candidates(roots):
    candidates = []
    for base in roots:
        base = Path(base)
        if not base.exists():
            continue
        for current, dirs, names in os.walk(base):
            here = Path(current)
            dirs[:] = [d for d in dirs if d not in
                       {".lake", ".git", "node_modules", "__pycache__", "primitive-lean-results"}]
            if (here / "lean/lean-toolchain").is_file():
                candidates.append(here)
                dirs[:] = []
            if "archive.zip" in names:
                candidates.append(here / "archive.zip")
            if len(here.relative_to(base).parts) >= 4:
                dirs[:] = []
            if len(candidates) >= 100:
                return candidates
    return candidates


class SafeRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        redirected = super().redirect_request(req, fp, code, msg, headers, newurl)
        if redirected is not None and urllib.parse.urlsplit(req.full_url).netloc != urllib.parse.urlsplit(newurl).netloc:
            redirected.remove_header("Authorization")
        return redirected


def github_download(repository, commit, secret_label, destination):
    """Read-only private archive download; no token in URL, log or Git config."""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(secret_label)
        request = urllib.request.Request(
            "https://api.github.com/repos/" + repository + "/zipball/" + commit,
            headers={"Authorization": "Bearer " + token, "Accept": "application/vnd.github+json",
                     "User-Agent": "primitive-lean-kaggle"})
        opener = urllib.request.build_opener(SafeRedirect())
        with opener.open(request, timeout=120) as response, open(destination, "wb") as target:
            total = 0
            while chunk := response.read(1024 * 1024):
                total += len(chunk)
                if total > 64 * 1024 * 1024:
                    raise ValueError("Source ZIP exceeds 64 MiB")
                target.write(chunk)
    except Exception:
        raise RuntimeError("GitHub取得に失敗。InternetとKaggle Secretsの読み取り権限を確認してください。") from None


def prepare_sources(config, expected, overlays, snapshot):
    root = Path(config["workspace"]).resolve()
    root.mkdir(parents=True, exist_ok=True)
    candidates = ([Path(config["source_path"])] if config["source_path"] else
                  find_candidates(config["search_roots"]))
    temporary = None
    if config["source_mode"] == "github":
        temporary = tempfile.TemporaryDirectory(prefix="primitive-github-")
        downloaded = Path(temporary.name) / "sources.zip"
        github_download(snapshot["repository"], snapshot["source_commit"], config["secret_label"], downloaded)
        candidates = [downloaded]
    elif config["source_mode"] not in {"auto", "path"}:
        raise ValueError("SOURCE_MODE must be auto, path or github")
    errors = []
    try:
        for candidate in candidates:
            try:
                files = read_candidate(candidate, expected)
                mismatches = source_matches(files, expected, overlays)
                if mismatches:
                    errors.append({"source": str(candidate), "missing_or_changed": mismatches[:8]})
                    continue
                source_root = candidate if candidate.is_dir() else None
                files.update({p: text.encode("utf-8") for p, text in overlays.items()})
                for rel, data in files.items():
                    target = root / rel
                    if not target.resolve().is_relative_to(root):
                        raise ValueError("Destination path escapes workspace")
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(data)
                hashes = {p: sha_bytes(data) for p, data in files.items()}
                if any(sha_bytes((root / p).read_bytes()) != digest for p, digest in hashes.items()):
                    raise ValueError("Copied source hash mismatch")
                cache = source_root / "lean/.lake/packages" if source_root is not None else None
                dest_cache = root / "lean/.lake/packages"
                reused = False
                if (config.get("reuse_cache", True) and cache is not None and cache.is_dir()
                        and str(cache.resolve()).startswith("/kaggle/working/")
                        and cache.resolve() != dest_cache.resolve() and not dest_cache.exists()):
                    dest_cache.parent.mkdir(parents=True, exist_ok=True)
                    dest_cache.symlink_to(cache.resolve(), target_is_directory=True)
                    reused = True
                return {"status": "passed", **snapshot, "source_subset_only": True,
                        "selected_input": "private GitHub snapshot" if temporary else str(candidate),
                        "verified_files": len(hashes), "sha256": hashes, "workspace": str(root),
                        "dependency_cache_reused": reused,
                        "adapter_overlay_paths": sorted(overlays), "lean_proofs": "not_run"}
            except (OSError, ValueError, zipfile.BadZipFile) as error:
                errors.append({"source": str(candidate), "error": str(error)})
    finally:
        if temporary is not None:
            temporary.cleanup()
    raise RuntimeError("対応する原本が見つかりません。SOURCE_PATHに原本フォルダ/ZIPを指定するか、archive.zipをInputへ追加してください。詳細: " + json.dumps(errors[:4], ensure_ascii=False))


def run_logged(args, cwd, logfile, timeout=1800):
    started = time.monotonic()
    print("実行:", " ".join(str(x) for x in args), flush=True)
    with open(logfile, "w", encoding="utf-8") as log:
        try:
            proc = subprocess.run(args, cwd=cwd, stdout=log, stderr=subprocess.STDOUT,
                                  text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            raise RuntimeError("時間上限に達しました。ログを確認してください: " + str(logfile)) from None
    row = {"command": [str(x) for x in args], "exit_code": proc.returncode,
           "seconds": round(time.monotonic() - started, 3), "log": str(logfile)}
    if proc.returncode:
        print(Path(logfile).read_text("utf-8", errors="replace")[-5000:])
        raise RuntimeError("コマンドが失敗しました。ログ: " + str(logfile))
    return row


def setup_lean(root, runtime, report_dir, installer):
    root, runtime, report_dir = Path(root), Path(runtime), Path(report_dir)
    runtime.mkdir(parents=True, exist_ok=True)
    project = root / "lean"
    manifest_bytes = (project / "lake-manifest.json").read_bytes()
    toolchain_bytes = (project / "lean-toolchain").read_bytes()
    os.environ.pop("ELAN_TOOLCHAIN", None)
    binaries = Path(os.environ.get("ELAN_HOME", str(Path.home() / ".elan"))) / "bin"
    os.environ["PATH"] = str(binaries) + os.pathsep + os.environ.get("PATH", "")
    runs = []
    if shutil.which("elan") is None:
        script = runtime / "elan-init.sh"
        try:
            with urllib.request.urlopen(installer["url"], timeout=120) as response:
                raw = response.read()
        except Exception:
            raise RuntimeError("Lean環境の取得に失敗。KaggleのInternetをOnにしてください。") from None
        if sha_bytes(raw) != installer["sha256"]:
            raise ValueError("Elan installer checksum mismatch")
        script.write_bytes(raw)
        runs.append(run_logged(["sh", str(script), "-y", "--default-toolchain", "none", "--no-modify-path"], runtime, report_dir / "elan-install.log"))
    runs.append(run_logged(["elan", "toolchain", "install", toolchain_bytes.decode().strip()], project, report_dir / "toolchain-install.log"))
    runs.append(run_logged(["lake", "env", "lean", "--version"], project, report_dir / "lean-version.log"))
    # Limit cache retrieval to the imported Mathlib modules and their closure.
    sources = list((project / "Elastica").glob("*.lean")) + list((root / "linear_input/lean_check").glob("*.lean")) + list((root / "linear_input/lean_check").glob("*.inc"))
    imports = sorted({m for p in sources for m in re.findall(r"(?m)^import\s+(Mathlib\.[A-Za-z0-9_.]+)", p.read_text("utf-8"))})
    if not imports:
        raise ValueError("No Mathlib imports found")
    runs.append(run_logged(["lake", "exe", "cache", "get", *imports], project, report_dir / "mathlib-cache.log", timeout=3600))
    if manifest_bytes != (project / "lake-manifest.json").read_bytes() or toolchain_bytes != (project / "lean-toolchain").read_bytes():
        raise ValueError("Pinned configuration changed during setup; do not continue")
    dependencies = []
    for package in json.loads(manifest_bytes)["packages"]:
        proc = subprocess.run(["git", "rev-parse", "HEAD"], cwd=project / ".lake/packages" / package["name"], capture_output=True, text=True)
        dependencies.append({"name": package["name"], "expected": package["rev"], "actual": proc.stdout.strip(), "match": proc.returncode == 0 and proc.stdout.strip() == package["rev"]})
    if not all(row["match"] for row in dependencies):
        raise ValueError("Pinned dependency revision mismatch")
    return {"status": "passed", "runs": runs, "dependencies": dependencies,
            "toolchain": toolchain_bytes.decode().strip(), "proofs": "not_run"}


def run_proofs(root, report_dir, expected_hashes, rebuild=False):
    root, report_dir = Path(root), Path(report_dir)

    def check_snapshot():
        changed = [p for p, digest in expected_hashes.items()
                   if not (root / p).is_file() or sha_bytes((root / p).read_bytes()) != digest]
        if changed:
            raise ValueError("Source snapshot changed: " + ", ".join(changed))

    check_snapshot()
    started_ns = time.time_ns()
    args = [sys.executable, "-X", "utf8", "tools/verify.py", "--proof-only"]
    if rebuild:
        args.append("--rebuild")
    run = run_logged(args, root, report_dir / "proof-run.log", timeout=10800)
    path = root / "verification/proof-only.json"
    if not path.is_file() or path.stat().st_mtime_ns < started_ns:
        raise ValueError("No fresh proof-only report")
    result = json.loads(path.read_text("utf-8"))
    if result.get("passed") is not True:
        raise ValueError("The current proof check did not pass")
    check_snapshot()
    return {"status": "passed", "run": run, "result": result}


def export_results(root, report_dir, state):
    root, report_dir = Path(root), Path(report_dir)
    save_json(report_dir / "notebook_run.json", state)
    since = state.get("proof_attempt_started_ns")
    if state["proofs"]["status"] in {"passed", "failed"} and since is not None:
        for rel in ["verification/proof-only.json", "verification/main-result.json", "verification/main-result.log",
                    "verification/proof-only-joint.log", "verification/proof-only-supplement.log",
                    "linear_input/lean_results.json", "linear_input/lean_check/build.log", "linear_input/lean_check/elastica-build.log"]:
            path = root / rel
            if path.is_file() and path.stat().st_mtime_ns >= since:
                dest = report_dir / "proofs" / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(path, dest)
    output = report_dir.with_suffix(".zip")
    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(report_dir.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(report_dir))
    return output

EXPECTED = {'AGENTS.md': '482d6847615fd78173f303d43e86b268d46d21f82b464d8e47401d7ccc5596c0', 'lean/Audit.lean': 'ed5b128debb363bd6becb9f5a79d18c6faec893c46cd24e4aad68e534f82a6eb', 'lean/Elastica/AffineEquivalence.lean': '5c1b4b92292a98200c5fda9e010e3b9083990523011f70c2de93977f5bad8d6b', 'lean/Elastica/CombinedConsequences.lean': '89c1dd68142e6d45dcf2af1a5eadf8860dc8b757327a658dc237ac7559a7b7b2', 'lean/Elastica/Composition.lean': '3834475a1b41cd6f1c89bb2c50c77e9701d6e7cb78aec740d0fe85476574879b', 'lean/Elastica/ConstrainedSecondVariation.lean': '0f6fef7f9a1b625f1662e77d413c6797415ac35d09ba31b59fb914f04dd8b01d', 'lean/Elastica/ConstraintRegularity.lean': 'cbc7af0cb4815acda77a7d363f716c7da0df0987ec7f665d18cc26003f9808fe', 'lean/Elastica/Derivation.lean': '760d16e7cd7106cdd48951721ea187fae6a79110a8d5ba665e50634a478c98dc', 'lean/Elastica/Equations.lean': '588563ce3b56c21eebfcfb9df36e8ad5b0e052897e689b6f90f673e75f6419fa', 'lean/Elastica/ExactVariation.lean': '76a8b04f8b8e05f4c876e5cd02968995e0b839cf1fff56fcdd46c62dd290cdbb', 'lean/Elastica/Examples.lean': 'c39ef5221465496305f9af9b62edd39d86d947dc5e7468e59a893a95398f4356', 'lean/Elastica/ExponentialWeight.lean': '0bf9e3d2c0f22ef7034dd87deac7baafe14ad7c83b113d815e22d81b7e8268ab', 'lean/Elastica/Geometry.lean': '5a7d021116a5ecc099f33f2cf4fbfc2189ff2b8053aa3065a10d4fcc4a1bdece', 'lean/Elastica/LocalMinimum.lean': 'b838b5c6291f8de3df85bf5fbe61ff0f83c19b596e259d07460d3d912411330c', 'lean/Elastica/MinimumConditions.lean': '2e1fac501040dd0dd2fd009040cd8119be2464328c1605d940eb5ca6d4876a53', 'lean/Elastica/PowerLogResponse.lean': '4c8cf9238a2752d78a3cde207f4aea97c8649259a5e3bca8baa70b02abe560da', 'lean/Elastica/Reconstruction.lean': '20e2c467cfabcad9e2d11669e75f2183b216a1effe090bcf58290a238d21a297', 'lean/Elastica/ReplacementConsequences.lean': 'f0a4fa40e6faffc640a26a9b1d3101d795c416add5cf0c70219bc811b6dc8d80', 'lean/Elastica/ScalarMinimum.lean': 'c372ccc55c7eff747c13d51f464f0c4c4003bd23e350c68f113022e32f99a789', 'lean/Elastica/Scaling.lean': 'b7c8078c62baf4e297d40db237dff1d71ec4245fc4f8cc812d4432735de254b8', 'lean/Elastica/ScalingMinima.lean': '6e7dc809acc1ebc49232c2500419a538b3012703df9cf5c9d0318a26600ef570', 'lean/Elastica/SecondVariation.lean': '2b592aedc1f4ec119196ac6a4ad04ff2376a3521d5f1276c014946d3b60bfcbf', 'lean/Elastica/Stationary.lean': 'fa36841f118f4500eff1dcb3ac7503c8a31cc0bf6a87909f0cb601587943d729', 'lean/Elastica/StraightConstraint.lean': '93cd7f280bb2f9bf28727dcc6fc474496f8f37e8db09b2cd04a995e32c0af66e', 'lean/Elastica/Variation.lean': '0816fa64e6d6b63c4cb8faa13114586f06a17928d2b948738e68c08276263e5f', 'lean/Elastica/WeightResponse.lean': '10e2084516c9e98d1b3f12feed1b0cd56c4dc9780056e3a0284fa9623f8ec60d', 'lean/Elastica.lean': 'b701498633383a7c42397eaa7c149801c673bd06cf808091b98ddfe7839607ed', 'lean/README.md': '426aab3497d68fb6e111287d438f93c951ee79fa2653eafacdcf094f02753b6b', 'lean/lake-manifest.json': '5781fd49aff59347f594d12cbfa61129eebb10ef293729ead611bd7410740590', 'lean/lakefile.toml': '24cbaef7acf55e1276267d5383d3107b695d5caa5ab0f12abd535c594c86eabe', 'lean/lean-toolchain': 'ce4c4e3d87434b9663f46de25ce34b48a0cf0d392e0a320a0787b4674a2d7b61', 'linear_input/capture_sources.py': '2b11148453e91ffaa75473bfa6615b33a368be4a087e0fc228949982d180b333', 'linear_input/check_delivery.py': '7b718a21860c50a3e7aaff67afcbfbf9f810ad050f98e7d6facb25beea6aeebb', 'linear_input/lean_check/AreaExactVariation.inc': '3c51bc3a0dd811b8f6c7ae0f0ce7c75aca9f7704a5852db6c89bbd7d76c8f1e9', 'linear_input/lean_check/AreaFamily.inc': '2ce60a18660fd4c4118d66edde22aca3b8f1e63264de44d396039fe658649160', 'linear_input/lean_check/AreaMinimum.inc': '6495ef140babfc06ab718b5d19f9567d939fdbc1d9d2aca4edd11a60eceb9afd', 'linear_input/lean_check/AreaParameters.inc': 'e23141b25aa88c8c7f9f4075327b317e28813aee3a652c07c4f0dfdfea36978b', 'linear_input/lean_check/AreaRegularity.inc': '5c7f22565f41884dfa56ee2236b083bf01940fae92a71a828cc72289c9a837a6', 'linear_input/lean_check/AreaVariation.inc': '0c6a571742fe0add0488fc917124dcfbb727c29cd1aa0e1881f82944743a60bb', 'linear_input/lean_check/Calculus.lean': '6adc1633283e928968d877db17ca58b647cf913d2ee68f41043e77a3fecc629a', 'linear_input/lean_check/Coercivity.inc': 'a1e6fd5b4b5278e8feb891bd6acdaf62774b01b21f5fbdd72079f94eb5317947', 'linear_input/lean_check/CompactLocalExistence.inc': 'ee81e16d1c8b7e7583c37a5dc1c525205894be20c9e526da7f4967b1ad5a7fab', 'linear_input/lean_check/Consequences.inc': 'b63d8fc479da5ae18bc0efa84bc4b99737f5e64ca64c7e733e82c566fe531ef6', 'linear_input/lean_check/Constraints.inc': '3f749b971efc58d0f4451c66c7eff38dcaf5bc37642a531225121931da528546', 'linear_input/lean_check/CoupledConsequences.inc': '370e8c125c0f0cef840398ececf6137f15d9665176fb18d76410588aadeb7db6', 'linear_input/lean_check/CoupledTrajectory.inc': 'e575924d3694a4cabaecde04a9f31266ed4a640e5d01c6ef1843de804653f70a', 'linear_input/lean_check/CurvatureBridge.inc': 'd23bc45b8bcd06650b3de879fbed45a532a37506cd8126673ea1dcab8189d79f', 'linear_input/lean_check/CurvatureRigidity.inc': 'b89461438d07ecdfd95c97e1a8f2a61dc6ecf05840808756f04d674d4640d0d5', 'linear_input/lean_check/Design.lean': 'e8ce22c9c6fadd68aeb9b33a9f7cf828ab58e9b5e8fc5f1b4494f5763f0cf549', 'linear_input/lean_check/Dissipation.lean': '2f98fe11dd728aa08038a1d3e01a9440ba83049b6dd4d3d1561a3dd140aac8e8', 'linear_input/lean_check/Feedback.inc': '9d97c366584d04da3fbdcd63080b2e98a7044f88132d83cfb8d6540002225747', 'linear_input/lean_check/FiniteBand.inc': '9d7d506703c5f9e816b3e1c6bd988bad8a697f8d25bbd7a1354b63d420ea18cd', 'linear_input/lean_check/FiniteInternalUpdate.inc': '8d919b3d39be880fd82959c7be1ee9dd40e9f5d2d009d70af95bbb7dc8e1b2b9', 'linear_input/lean_check/ForwardUniqueness.inc': 'a93efd6d16caf0d173b5d71ae872bbff50ae113ce8a6ff8dbd68dbbb0605f214', 'linear_input/lean_check/GeneralMinimum.inc': 'ce31350415300a6f772521c13d89f61808e4ed4911d7c37f39ac2bafde1987ca', 'linear_input/lean_check/GlobalConsequences.inc': 'fd0dbd3ca5030b1cce1151bb5f11c0450b0fba7df6663be000732cdd4b7a1969', 'linear_input/lean_check/GlobalContinuation.inc': 'f40e5840fa9ec00e5900968836b1c1868c05bfce559258ffd6e2259c08145664', 'linear_input/lean_check/GlobalTrajectory.inc': '67b5494e57c0f9a9b4020486f4ea59468a27001601c2a4b34f84157b3f131ba9', 'linear_input/lean_check/Gradient.inc': '2aab575d80c5eca3871b68ac141b179ef7ab072e071097d3dfe758b13bf96201', 'linear_input/lean_check/GradientBound.inc': 'e1468dc3a1d2647923debe57854f0e77c745d88244daf44be2b1b890e8e9a770', 'linear_input/lean_check/InternalUpdate.inc': '6493eb6ce9d576bb789f96fb33eef6a97ba0d2bb7100d9db8985cc0b63c7d63a', 'linear_input/lean_check/Invariance.inc': '7ba29d42bdde5903ebb47db238afefe3864fe1ea1580812cb1cbd2f16f5f0465', 'linear_input/lean_check/LevelConvergence.inc': '268cdcb7da22ea5e9ea9529106e1a2f8970b8031cd7e0161073380122ad1de3f', 'linear_input/lean_check/LevelGeometry.inc': '69bbfdb3ef02bf3b8c43f9388bda46a3d52bd29519c15752ac0ba84a57b53b63', 'linear_input/lean_check/MainResult.inc': '7bded7415fafff042a23164fd4f12aa053dd5595e99b878b02e6ca4a5eac45da', 'linear_input/lean_check/Observation.lean': '2a817b9f8550e8111bce7cab59ecccfce1a56b852d442d8012c32e523271ebc6', 'linear_input/lean_check/PressureCurvature.inc': '4a0365823e7eec686dd940e531987ada928cb0e19ca6901d440745ce9da221b1', 'linear_input/lean_check/PressureStationary.inc': '4b95864614e426282726e16fbfe16c06c86f55e563fa1f7085060f606f0d0c83', 'linear_input/lean_check/QuarticClassification.inc': '1a733d21a58ecf03e26bd06ecc8fb79e25541e76762e4c1c1b20e5acec3fc89d', 'linear_input/lean_check/README.md': '424a4faff3fd83424a539ff030d207e208ecc2b611b022a2ee20c70a2e08e116', 'linear_input/lean_check/RegularBand.inc': '53d0053d7d4f0bf31eedc9a8e322c246a4b04f495f68c0d418084cc4c77cd6a4', 'linear_input/lean_check/StrictEnergy.inc': '1a701960dd2d6118175f467a55ebf2a6902bac81581496a1dbe59d8d680cba7f', 'linear_input/lean_check/SupportComposition.inc': '0ed4c127f19fcb5eaa9512013a9ead4bcb723455d831c1a371dc62893b04392c', 'linear_input/lean_check/TrajectoryExistence.inc': 'db5f37cc4923fd87063a75abc3ae560cd135ebe7c4d79104bd472f22d2dd53b0', 'linear_input/lean_check/VariationalBridge.inc': '5b7b6619add1121a5245ef40f25b1be960682c8aa8617bb2accd19f10ebffb02', 'linear_input/linear_input.py': 'd7a157aea20aa26c44a119ac94b4c1d2e12831069b622049a401e4aa4f5ebb8a', 'linear_input/pressure_sources.json': 'b0ec046ddc9218fe5ab5dc9492c04e87bbff39f0de82b61143d32bda97077132', 'linear_input/provenance/affine_equivalence_excerpt.txt': '9b085b45d587318d97070dafe91823c76891e52c4cd385ecd4612dcbc5ff7550', 'linear_input/provenance/energy_findings_excerpt.txt': '17e1933472e7407b916d1bea6cf517065e8936fa265a5f0a134b6d99aee20d75', 'linear_input/provenance/power_log_excerpt.txt': 'a8fe681da590fe8681a54eee0d6e44696a7ec9b98352eac98a280abbd44d69c1', 'linear_input/provenance/primitive_operations_excerpt.txt': 'c6472e938b449ea4ddd436fd006d80f3edc2feb1b592298864aafc6b0231359c', 'linear_input/render.py': '24166b4d1bc7f4bb697f8fc0da53dde855b59df8aa9745dac9c28375bcc52124', 'linear_input/sources.json': 'ab9bd32198e9c11ec48ce8264e4c66308a5577642ddb779a56062cb147f72d8d', 'linear_input/verify.py': '6c718f7883c4ee41eee1e452c9ba89da230cd21f2156c378ecfc6416aada3ecb', 'linear_input/verify_lean.py': 'f22428e8bf856a165d7ec43c9dac05ebd6c6d59380c27b893d135f1d38afd9b6', 'linear_input/線形入力の使い方.md': 'b4921b652de0dbb4a2e7785f79e0e6593835384e5bda19ab379b2572edecf52d', 'requirements.txt': 'e5f223cf307a68a2b0f25aba3e415f62caa06e2fdd7cf008f901af18dd6e31aa', 'sources/general_minimum_2026-10-21.json': '55b73a624fc52f0de096694309c68a258913ef5d355337b8ee5eddb5cc68b6d0', 'sources/provenance.json': '2293ad47669ac50f45e0af28bdc53331ccfab86826550d74cdf19a8a9d47f1cc', 'sources/research_01.txt': 'f92ddad8f38b49a5f876588610f572c82fa670ec5b1194d667dc18e8b84f97a4', 'sources/research_02.txt': '0a4ffc9a7ef5ba5492564c42ab712aa834a1709f75f3abd9b87d1420ea4b9bc8', 'sources/research_03.txt': '3d4c154f3a49d0de88b12720771cc6734ce0a696fa417f09cd6f3fb29feeae49', 'sources/research_04.txt': '1ac7d6eb10a1d51cebaf485f5e8d668828c0851703948c8c83174824d402091b', 'sources/research_04_2026-10-21.txt': '8ea8b36c40435f0a93990101f59e4edfcb95c18a1d40f43f82411757f1eecf3d', 'sources/research_05.txt': 'a4b1a5971fe06e72d481460bc8c08d56b40a5a16d3e7d11c4f6e1f133e97502f', 'sources/research_06.txt': '2e43ab064c2f21cd560d0f9d1268324f7148974caf6b86a79bd2ee538cbcae02', 'sources/research_07.txt': '0e4c47e39a2a685ac45aacf2f5043860007972ef8b694da1dc41fc9d62d36d01', 'sources/research_08.txt': 'eb69ab3fdef48fda428429fe58ee8431a83535f0491152ca9b14c14adb9b5234', 'sources/research_09.txt': '468531a01f71542d37e73975398ebca815662bb12df79d4d03331a565ecb2a4e', 'sources/research_10.txt': 'c64ce1f75d808d752986873465d20f70deadd4c840e7a5f94a5b8916bede7804', 'sources/research_11.txt': '9f2215533e6d626998eb88499c4e38193b5cb3d8dc7f0b8aafb6754e0c6375b3', 'sources/theorem_composition_2026-10-21.json': '9f87f211f14ac80c5fedd8f0232f8cbebc2b503a3c291fae8f801abf7a895cfd', 'sources/採用方針.txt': '0ff76585e0e3002b2ad10058522c81af98e4d74d9c232e4778f57d0593e8133d', 'tools/capture_additions.py': '923e47ce77b6160a3a607ef98d4cbe857fa8758706194b6267d8d8b76f5f79c5', 'tools/capture_sources.py': '74dcd2003c41897f7a19086b5686bdf9cd0dbaa266d188b14fc3e6feda964db2', 'tools/check_combinations.py': 'be9c925976aa9f6379bb99ac0b70bbf06037db9bcbf408803af8bafee8e0de23', 'tools/check_delivery.py': '73d07fed611de15a8b571a1e3eb74e1fd203374d0834cd1de3b608dbbf2b5c49', 'tools/check_examples.py': 'a98bf45972cb351cd962c7eb10259a27ed464d3cd5938f9ec413593bbc355834', 'tools/check_scale_response.py': 'c3d988f0f636936626aede21726210f1981967c00b5256659538defc8c1a2f3f', 'tools/render.py': 'b4da92885485cea04cc0b647c66c53337c4a2eecfe4b2df1e3040dbf0247b664', 'tools/render_math.cjs': 'aa07fbfd2377b91a1273f604735e1a4645d0345065b49d85a2267f1c431c6f7a', 'tools/verify.py': '65654fc1abab6a2bc5576d585647ea2ec35533d597f0d57a82e79c4ffe84fe72', 'tools/verify_main_result.py': '423070324aa6f5a362b5cf38fd655ccd625b0da8c0b1bc312777f6154749460c', 'verification/before-unification.json': 'de42e2a051a3e24ea4db696fc771b1a25ae314ac72efa1b263192963df08f98f', '原始の形式化.md': '334d7e140b5ffa2f204a2fd7f862d5365fa15713404b0602ec3e2982605e2167', '検証と再実行.md': '7ef4e463908b0defa267e1a959eddb8c5873f2c0e8297ad7acaf04f27d9096d7'}
OVERLAYS = {'tools/verify.py': '"""Verify retained proof groups and additive compositions through one entry point.\r\n\r\nUsage: python -X utf8 tools/verify.py [--rebuild] [--lean-only | --proof-only]\r\n--rebuild removes only lean/.lake/build; pinned dependency caches are retained.\r\n--proof-only checks fresh proofs without requiring the separate historical archive audit.\r\n--lean-only runs the unchanged joint checker, then audits all registered\r\nsupplements together with all original declarations.\r\n"""\r\nfrom pathlib import Path\r\nfrom datetime import datetime, timezone\r\nimport argparse, hashlib, json, re, shutil, subprocess, sys, time, zipfile\r\n\r\nROOT = Path(__file__).resolve().parents[1]\r\nLEAN = ROOT / "lean"\r\nOUT = ROOT / "verification"\r\nALLOWED = {"propext", "Classical.choice", "Quot.sound"}\r\n\r\ndef sha(path):\r\n    return hashlib.sha256(path.read_bytes()).hexdigest()\r\n\r\ndef write_json(path, value):\r\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\r\n\r\ndef strip_comments_strings(source):\r\n    """Keep source line offsets while excluding nested comments and strings."""\r\n    result=[]; i=0; depth=0; string=False\r\n    while i<len(source):\r\n        c=source[i]; pair=source[i:i+2]\r\n        if depth:\r\n            if pair==\'/-\':depth+=1;result.extend(\'  \');i+=2;continue\r\n            if pair==\'-/\':depth-=1;result.extend(\'  \');i+=2;continue\r\n            result.append(\'\\n\' if c==\'\\n\' else \' \');i+=1;continue\r\n        if string:\r\n            if c==\'\\\\\':result.extend(\'  \');i+=2;continue\r\n            if c==\'"\':string=False\r\n            result.append(\'\\n\' if c==\'\\n\' else \' \');i+=1;continue\r\n        if pair==\'/-\':depth=1;result.extend(\'  \');i+=2;continue\r\n        if pair==\'--\':\r\n            end=source.find(\'\\n\',i);end=len(source) if end<0 else end\r\n            result.extend(\' \'*(end-i));i=end;continue\r\n        if c==\'"\':string=True;result.append(\' \');i+=1;continue\r\n        result.append(c);i+=1\r\n    return \'\'.join(result)\r\n\r\ndef command(args, name, cwd=ROOT):\r\n    started=time.monotonic()\r\n    proc=subprocess.run(args,cwd=cwd,text=True,encoding="utf-8",errors="replace",capture_output=True)\r\n    output=proc.stdout+proc.stderr\r\n    (OUT/(name+".log")).write_text(output,encoding="utf-8")\r\n    warnings=[line for line in output.splitlines() if re.search(r"\\bwarning:",line)]\r\n    return dict(command=args,working_directory=str(cwd),exit_code=proc.returncode,\r\n                seconds=round(time.monotonic()-started,3),log="verification/"+name+".log",\r\n                warnings=warnings)\r\n\r\ndef canonical_path(relative):\r\n    return (ROOT / "linear_input" / relative).resolve().relative_to(ROOT).as_posix()\r\n\r\n\r\ndef verify_lean(rebuild):\r\n    if rebuild:\r\n        target = (LEAN / ".lake/build").resolve()\r\n        if target != LEAN.resolve() / ".lake/build":\r\n            raise RuntimeError("Unsafe project build path")\r\n        if target.exists():\r\n            shutil.rmtree(target)\r\n    # The proof files and their existing checker retain their separate roles.\r\n    run = command([sys.executable, "-X", "utf8", "linear_input/verify_lean.py"], "joint-lean-run")\r\n    current = json.loads((ROOT / "linear_input/lean_results.json").read_text("utf-8"))\r\n    original_declarations = [{**d, "file": canonical_path(d["file"])}\r\n                    for d in current["parent_declarations"] + current["declarations"]]\r\n    before = json.loads((OUT / "before-unification.json").read_text("utf-8"))\r\n    archive_path = ROOT / before["archive"]\r\n    assert sha(archive_path) == before["archive_sha256"]\r\n    with zipfile.ZipFile(archive_path) as archive:\r\n        old = json.loads(archive.read("linear_input/lean_results.json"))\r\n        old_names = {d["name"]: d["kind"] for d in old["parent_declarations"] + old["declarations"]}\r\n        new_names = {d["name"]: d["kind"] for d in original_declarations}\r\n        proof_files = sorted(set(d["file"] for d in original_declarations))\r\n        unchanged = [rel for rel in proof_files if (ROOT / rel).read_bytes() == archive.read(rel)]\r\n    preservation = dict(passed=len(unchanged) == len(proof_files) and old_names == new_names,\r\n        proof_files=len(proof_files), byte_identical_proof_files=len(unchanged),\r\n        same_public_names_and_kinds=old_names == new_names,\r\n        missing=sorted(old_names.keys() - new_names.keys()), added=sorted(new_names.keys() - old_names.keys()),\r\n        archive=before["archive"])\r\n    warnings = []\r\n    for rel in (current["compile"]["log"], current["parent_library_build"]["log"]):\r\n        warnings += [line for line in (ROOT / "linear_input" / rel).read_text("utf-8").splitlines()\r\n                     if re.search(r"\\bwarning:", line)]\r\n    supplement_run = command([sys.executable, "-X", "utf8", "tools/verify_main_result.py"], "main-result-run")\r\n    supplement_path = OUT / "main-result.json"\r\n    supplement = json.loads(supplement_path.read_text("utf-8"))\r\n    declarations = original_declarations + supplement["declarations"]\r\n    axioms = supplement["axioms"]\r\n    hashes = {canonical_path(rel): digest for rel, digest in current["source_sha256"].items()}\r\n    hashes.update(supplement["source_hashes"])\r\n    hashes["tools/verify.py"] = sha(Path(__file__))\r\n    hashes["verification/main-result.json"] = sha(supplement_path)\r\n    warnings += supplement["warnings"]\r\n    changed = [rel for rel, digest in hashes.items() if not (ROOT / rel).is_file() or sha(ROOT / rel) != digest]\r\n    passed = (run["exit_code"] == 0 and current["passed"] and preservation["passed"]\r\n              and supplement_run["exit_code"] == 0 and supplement["passed"]\r\n              and supplement["all_declarations"] == declarations\r\n              and len(axioms) == len(declarations) and not warnings and not changed)\r\n    supplemental_scopes = {\r\n        "MainResult": "Additive connection and worked example",\r\n        "GeneralMinimum": "General fixed-input minimum and strict-support criteria, with quartic bridges",\r\n        "SupportComposition": "Support interpolation, ordering, derivative identification and shared-minimizer restrictions",\r\n        "QuarticClassification": "Exact quartic minimum uniqueness and supporting-input classification",\r\n        "CurvatureRigidity": "Conserved phase energy and the transverse square force constant curvature at a phase minimum",\r\n    }\r\n    supplemental_groups = {\r\n        Path(group["source"]).stem: dict(\r\n            theorem_count=group["theorem_count"], declaration_count=group["declaration_count"],\r\n            path=group["source"], scope=supplemental_scopes[Path(group["source"]).stem])\r\n        for group in supplement["supplemental_groups"]\r\n    }\r\n    result = dict(timestamp_utc=datetime.now(timezone.utc).isoformat(), passed=passed,\r\n        proof_layout="Existing Elastica and linear-input proofs retained; registered additions and theorem compositions audited together",\r\n        entry_point="tools/verify.py", detailed_joint_report="linear_input/lean_results.json",\r\n        supplemental_report="verification/main-result.json", supplemental_run=supplement_run,\r\n        version=current["version"], clean_project_build=rebuild, dependency_cache_retained=True,\r\n        run=run, build=current["parent_library_build"], baseline_audit=current["compile"], audit=supplement["compile"],\r\n        declarations=declarations, theorem_count=current["theorem_count"] + current["parent_theorem_count"] + supplement["theorem_count"],\r\n        declaration_count=len(declarations), axioms=axioms, joint_axiom_audit=supplement["joint_axiom_audit"],\r\n        proof_groups=dict(Elastica=dict(theorem_count=current["parent_theorem_count"],\r\n            declaration_count=current["parent_declaration_count"], path="lean/Elastica"),\r\n            LinearInput=dict(theorem_count=current["theorem_count"], declaration_count=current["declaration_count"],\r\n            path="linear_input/lean_check", scope="Unchanged original 36 proof parts"),\r\n            **supplemental_groups),\r\n        added_declarations=supplement["declarations"],\r\n        dependencies=current["dependencies"], source_hashes=hashes,\r\n        unexpected=current["unexpected"] + supplement["unexpected"] + supplement["errors"],\r\n        forbidden_tokens=current["forbidden_tokens"] + supplement["forbidden_tokens"],\r\n        warnings=warnings, changed_during_verification=changed,\r\n        pre_unification_preservation=preservation,\r\n        scope="The unchanged linear-input verifier builds Elastica, generates All.lean from the original 36 parts and audits the original 509 declarations. The additive checker then compiles the exact All.lean followed by every registered supplement with --trust=0 and audits all original and additional public declarations in one session. Supplements retain separate source groups and counts. Original proof files, names and configuration are preserved.",\r\n        not_covered=[\r\n            ("existence for arbitrary prescribed boundary/area data and regularity of weak "\r\n             "area-constrained minimizers; the explicit circle comparison minimum is covered")\r\n            if item == "existence and regularity of area-constrained minimizing curves" else item\r\n            for item in current["not_covered"]])\r\n    write_json(OUT / "lean.json", result)\r\n    # Keep familiar top-level log links, copying actual outputs rather than relabelling old runs.\r\n    shutil.copyfile(ROOT / "linear_input" / current["parent_library_build"]["log"], OUT / "lean-build.log")\r\n    shutil.copyfile(OUT / "main-result.log", OUT / "lean-axioms.log")\r\n    if rebuild and passed:\r\n        shutil.copyfile(OUT / "lean-build.log", OUT / "clean-rebuild.log")\r\n        write_json(OUT / "clean-rebuild.json", dict(timestamp_utc=result["timestamp_utc"],\r\n            clean_project_build=True, dependency_cache_retained=True,\r\n            build=current["parent_library_build"], cleaned_by="tools/verify.py --rebuild before the joint checker",\r\n            build_log="verification/clean-rebuild.log", build_log_sha256=sha(OUT / "clean-rebuild.log"),\r\n            source_hashes=hashes, dependencies=current["dependencies"]))\r\n    return result\r\n\r\n\r\ndef refresh_manifest():\r\n    files = {}\r\n    for path in sorted(ROOT.rglob("*")):\r\n        rel = path.relative_to(ROOT)\r\n        if not path.is_file() or ".lake" in rel.parts or "__pycache__" in rel.parts or path.name == "manifest.json":\r\n            continue\r\n        files[rel.as_posix()] = sha(path)\r\n    write_json(ROOT / "manifest.json", dict(algorithm="SHA256", files=files))\r\n\r\n\r\ndef main(argv=None):\r\n    parser = argparse.ArgumentParser(description=__doc__)\r\n    parser.add_argument("--rebuild", action="store_true")\r\n    modes = parser.add_mutually_exclusive_group()\r\n    modes.add_argument("--lean-only", action="store_true")\r\n    modes.add_argument("--proof-only", action="store_true")\r\n    args = parser.parse_args(argv)\r\n    OUT.mkdir(exist_ok=True)\r\n    if args.proof_only:\r\n        from proof_only import run_proof_only\r\n        proof = run_proof_only(ROOT, command, args.rebuild)\r\n        print(json.dumps(proof, ensure_ascii=False, indent=2))\r\n        return 0 if proof["passed"] else 1\r\n    lean = verify_lean(args.rebuild)\r\n    summary = {k: lean[k] for k in ("passed", "proof_layout", "version", "clean_project_build",\r\n        "theorem_count", "declaration_count", "proof_groups", "unexpected", "warnings", "forbidden_tokens")}\r\n    if args.lean_only:\r\n        print(json.dumps(summary, ensure_ascii=False, indent=2))\r\n        return 0 if lean["passed"] else 1\r\n    checks = {}\r\n    for key, script in (("examples", "tools/check_examples.py"), ("scale_response", "tools/check_scale_response.py"),\r\n                       ("combinations", "tools/check_combinations.py"), ("linear_input", "linear_input/verify.py"),\r\n                       ("rendering", "tools/render.py")):\r\n        checks[key] = command([sys.executable, "-X", "utf8", script], key.replace("_", "-") + "-run")\r\n    write_json(OUT / "results.json", dict(passed=False, status="delivery validation pending"))\r\n    write_json(ROOT / "manifest.json", dict(algorithm="SHA256", status="validation pending", files={}))\r\n    delivery_run = command([sys.executable, "-X", "utf8", "tools/check_delivery.py"], "delivery-run")\r\n    delivery = json.loads((OUT / "delivery.json").read_text("utf-8"))\r\n    passed = lean["passed"] and all(check["exit_code"] == 0 for check in checks.values()) and delivery_run["exit_code"] == 0 and delivery["passed"]\r\n    result = dict(timestamp_utc=datetime.now(timezone.utc).isoformat(), passed=passed, lean_passed=lean["passed"],\r\n        theorem_count=lean["theorem_count"], declaration_count=lean["declaration_count"],\r\n        clean_project_build=args.rebuild, **checks, delivery_run=delivery_run, delivery=delivery,\r\n        source_hashes=lean["source_hashes"])\r\n    write_json(OUT / "results.json", result)\r\n    refresh_manifest()\r\n    print(json.dumps({**summary, "passed": passed, "checks": checks, "delivery_passed": delivery["passed"]}, ensure_ascii=False, indent=2))\r\n    return 0 if passed else 1\r\n\r\n\r\nif __name__ == "__main__":\r\n    raise SystemExit(main())\r\n', 'tools/proof_only.py': '"""Fresh Lean checks without the separate historical-delivery audit.\n\nCalled only through tools/verify.py --proof-only. Proof sources and the two\nexisting checkers are unchanged. This mode does not certify archived releases.\n"""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport json\nimport shutil\nimport sys\nimport time\n\n\ndef run_proof_only(root, command, rebuild=False):\n    root = Path(root).resolve()\n    out = root / "verification"\n    out.mkdir(exist_ok=True)\n    result_path = out / "proof-only.json"\n    result = dict(\n        timestamp_utc=datetime.now(timezone.utc).isoformat(), passed=False,\n        scope="Fresh Elastica build and joint Lean proof/axiom checks only",\n        historical_preservation="not_checked", delivery_and_rendering="not_checked",\n        build_requested=True, clean_project_build=rebuild, stages=[], errors=[],\n        theorem_count=None, declaration_count=None,\n    )\n\n    def save():\n        result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\\n", "utf-8")\n\n    def retire(path):\n        if path.exists():\n            history = out / "proof-history"\n            history.mkdir(exist_ok=True)\n            path.rename(history / (path.stem + "-" + str(time.time_ns()) + path.suffix))\n\n    retire(result_path)\n    save()\n    try:\n        if rebuild:\n            target = (root / "lean/.lake/build").resolve()\n            if target != root / "lean/.lake/build":\n                raise ValueError("Refusing to clean an external/symlinked build directory")\n            if target.exists():\n                shutil.rmtree(target)\n        stages = (\n            ("joint", "linear_input/verify_lean.py", root / "linear_input/lean_results.json"),\n            ("supplement", "tools/verify_main_result.py", out / "main-result.json"),\n        )\n        for name, script, report_path in stages:\n            # A previous successful report must never stand in for this run.\n            retire(report_path)\n            run = command([sys.executable, "-X", "utf8", script], "proof-only-" + name, cwd=root)\n            row = dict(name=name, run=run, fresh_report=report_path.is_file(), passed=False)\n            result["stages"].append(row)\n            if report_path.is_file():\n                report = json.loads(report_path.read_text("utf-8"))\n                row["passed"] = run["exit_code"] == 0 and report.get("passed") is True\n                row["report"] = report_path.relative_to(root).as_posix()\n                row["warnings"] = run.get("warnings", [])\n                if row["warnings"]:\n                    row["passed"] = False\n                if name == "supplement" and row["passed"]:\n                    result["theorem_count"] = report["total_theorem_count"]\n                    result["declaration_count"] = report["total_declaration_count"]\n            save()\n            if not row["passed"]:\n                raise ValueError(name + " checker failed or did not produce a fresh passing report")\n        result["passed"] = True\n    except (OSError, ValueError, KeyError, TypeError) as error:\n        result["errors"].append(str(error))\n    save()\n    return result\n'}
SNAPSHOT = {'repository': 'Sumuji-zkdwn/primitive-lean', 'source_commit': '2e058f2d76e7c7d1fb0f811bcb26a990c47b10d9', 'original_import_commit': 'b358f1a2753637992cdcfbaa5f354db3562ecc46', 'support_sha256': '84b5a9c56a3b6f141715ba10e9028d054112d1b114dc1f286dd8d71dbad8bb6e'}
INSTALLER = {'url': 'https://raw.githubusercontent.com/leanprover/elan/9605b24b08cf68b78dce0014b6fa01be1f8e2cee/elan-init.sh', 'sha256': 'a620ff1641616222c8d37c54845492004bb84d6877cdbc944dd65c1aa685bf53'}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + str(time.time_ns())[-6:]
REPORT_DIR = Path(WORKSPACE).parent / "primitive-lean-results" / RUN_ID
REPORT_DIR.mkdir(parents=True, exist_ok=True)
STATE = {"run_id": RUN_ID, "source_commit": SNAPSHOT["source_commit"],
         "source": {"status": "not_run"}, "setup": {"status": "not_run"},
         "proofs": {"status": "not_run"},
         "historical_preservation": "not_checked", "delivery_and_rendering": "not_checked"}

def show_output(path):
    try:
        from IPython.display import display, FileLink
        display(FileLink(str(path)))
    except ImportError:
        print("結果ファイル:", path)

def notebook_stage(name, action):
    STATE[name] = {"status": "running"}
    save_json(REPORT_DIR / "notebook_run.json", STATE)
    try:
        STATE[name] = action()
    except Exception as error:
        STATE[name] = {"status": "failed", "error": str(error)}
        show_output(export_results(WORKSPACE, REPORT_DIR, STATE))
        raise
    save_json(REPORT_DIR / "notebook_run.json", STATE)
    print(name, STATE[name]["status"])


## 1. 原本とGitHubの対応確認
元の証明原稿は変更せず、統合入口のアダプター2ファイルだけNotebookに同梱した固定版を適用します。117ファイルを再照合し、対応するソースコミットを記録します。

In [ ]:
config = {"workspace": WORKSPACE, "source_mode": SOURCE_MODE, "source_path": SOURCE_PATH,
          "search_roots": SOURCE_SEARCH_ROOTS, "secret_label": GITHUB_SECRET_LABEL,
          "reuse_cache": REUSE_EXISTING_CACHE}
notebook_stage("source", lambda: prepare_sources(config, EXPECTED, OVERLAYS, SNAPSHOT))
print("対応コミット:", SNAPSHOT["source_commit"])
print("一致したファイル:", STATE["source"]["verified_files"])
print("依存キャッシュ再利用:", STATE["source"]["dependency_cache_reused"])


## 2. Lean環境の準備
`RUN_SETUP=True` のときだけ実行します。原本のLean・mathlib revisionを保持し、依存revisionを照合します。初回は取得に時間がかかります。

In [ ]:
if RUN_SETUP:
    if STATE["source"]["status"] != "passed":
        raise RuntimeError("先に段階1を完了してください")
    notebook_stage("setup", lambda: setup_lean(WORKSPACE, Path(WORKSPACE).parent / "primitive-runtime", REPORT_DIR, INSTALLER))
else:
    print("環境準備: 未実行（RUN_SETUP=False）")


## 3. 実際の証明・公理検査
元の統合入口 `tools/verify.py --proof-only` から、既存のjoint checker、続いて5追補のcheckerを実行します。前回の合格結果を流用せず、今回生成されたレポートと終了コードを確認します。

In [ ]:
if RUN_PROOFS:
    if STATE["setup"]["status"] != "passed":
        raise RuntimeError("RUN_SETUP=Trueにして段階2も実行してください")
    STATE["proof_attempt_started_ns"] = time.time_ns()
    notebook_stage("proofs", lambda: run_proofs(WORKSPACE, REPORT_DIR, STATE["source"]["sha256"], REBUILD))
    print("今回の検査対象:", STATE["proofs"]["result"]["theorem_count"], "定理 /",
          STATE["proofs"]["result"]["declaration_count"], "宣言")
else:
    print("証明検査: 未実行（RUN_PROOFS=False）")


## 結果を保存
段階別の成否、ソースハッシュ、実際の実行ログをZIPにまとめます。ソースZIP、キャッシュ、認証情報は含めません。失敗時にも、その時点の記録を保存します。

In [ ]:
output = export_results(WORKSPACE, REPORT_DIR, STATE)
print(json.dumps({name: STATE[name]["status"] for name in ("source", "setup", "proofs")}, ensure_ascii=False, indent=2))
show_output(output)
